In [ ]:
%load_ext autoreload
%autoreload 2

import anndata as ad
adata = ad.read_h5ad("./data/larry/larry_processed.h5ad")
adata

In [ ]:
import numpy as np
import pandas as pd
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import StandardScaler

# 1. Get the matrix
# scVelo and Scanpy usually run PCA on log-transformed spliced counts
X = adata.layers["spliced"].copy()

# 2. Subset to highly variable genes
hvg_mask = adata.var["highly_variable"].values
X = X[:, hvg_mask]

# 3. Log1p transform (Scanpy's default)
X = X.toarray() if hasattr(X, "toarray") else X
X = np.log1p(X)

# 4. Center and scale (Scanpy centers but doesn’t always scale variance)
scaler = StandardScaler(with_mean=True, with_std=False)
X = scaler.fit_transform(X)

In [ ]:
from scripts.VectorFieldEmbedder import VectorFieldEmbedder
from scripts.plotting import plot_velocity_streamplot
from scipy.sparse import issparse

# ---------- 1. Load full dataset ----------

V = adata.layers["velocity"]
V = V / np.std(V, axis=0, keepdims=True)

# UMAP parameters
umap_params = {
    "min_dist": 0.3
}

# ---------- 2. Initialize embedder ----------
emb = VectorFieldEmbedder(
    X, V,
    dist_method="phase",
    dof=100,
    method="umap",
    embed_kwargs=umap_params,
    alpha=0,
    max_tps_points=4000,  # ensures sampling limit is respected
    knn_k=30              # k-neighbors used in PhaseDistanceGraphSolver
)

emb.initialize_embedding()

# ---------- 3. Plot velocity streamplot ----------
plot_velocity_streamplot(
            X_2d=emb.X_emb,
            tps_vf=emb.tps_vf,
            scatter_color=list(adata.obs["state_info"].values),
            grid_density=1,
            stream_density=1.2,
            scatter_size=20,
            scatter_alpha=0.3,
            figsize=(6, 6),
            aspect=1,
            vmin=0.0,
            vmax=1.0,
            cmap="tab10",
            grid_size=30,
            show_labels=False
        )

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

# Extract unique labels
cell_labels = np.array(adata.obs["state_info"].values)
unique_labels = np.unique(cell_labels)

# Manual override
manual_color = {"Undifferentiated": "lightgrey"}

# Get all other labels
other_labels = [label for label in unique_labels if label != "Undifferentiated"]

# Create color map using tab10
tab10 = plt.get_cmap("tab10", len(other_labels))
tab_colors = {label: mcolors.to_hex(tab10(i)) for i, label in enumerate(other_labels)}

# Combine
custom_colors = {**tab_colors, **manual_color}

# Build color array
cell_colors = np.array([custom_colors[label] for label in cell_labels])

# Assume X_emb is your 2D embedding
X_emb = emb.X_emb

# Create mask for undifferentiated cells
undiff_mask = cell_labels == "Undifferentiated"

# 1. Plot Undifferentiated first (background layer)
plt.figure(figsize=(6, 6))
plt.scatter(
    X_emb[undiff_mask, 0], X_emb[undiff_mask, 1],
    c="lightgrey", s=10, alpha=0.5, linewidths=0
)

# 2. Plot other cells on top
plt.scatter(
    X_emb[~undiff_mask, 0], X_emb[~undiff_mask, 1],
    c=cell_colors[~undiff_mask], s=10, alpha=0.5, linewidths=0
)

plt.axis("off")

# Create legend handles (excluding "Undifferentiated")
handles = [
    plt.Line2D([], [], marker='o', linestyle='', color=color, label=label, markersize=6)
    for label, color in custom_colors.items() if label != "Undifferentiated"
]

# Show legend
plt.legend(handles=handles, fontsize=8, loc='center left', bbox_to_anchor=(1.02, 0.5), borderaxespad=0.)

In [ ]:
from scripts.plotting import compute_velocity_on_grid

# === 1. Prepare embedding and labels ===
X_emb = emb.X_emb
cell_labels = np.array(adata.obs["state_info"].values)

# === 2. Define custom colors ===
unique_labels = np.unique(cell_labels)
manual_color = {"Undifferentiated": "lightgrey"}
other_labels = [label for label in unique_labels if label != "Undifferentiated"]
tab10 = plt.get_cmap("tab10", len(other_labels))
tab_colors = {label: mcolors.to_hex(tab10(i)) for i, label in enumerate(other_labels)}
custom_colors = {**tab_colors, **manual_color}
cell_colors = np.array([custom_colors[label] for label in cell_labels])

# === 3. Compute velocity on grid ===
Xg_all, _, keep = compute_velocity_on_grid(X_emb, grid_size=30)
Xg = Xg_all[keep]
Vg = emb.tps_vf.predict(Xg)

# === 4. Plot scatter and quiver ===
plt.figure(figsize=(6, 6))

# Plot Undifferentiated first (background layer)
undiff_mask = cell_labels == "Undifferentiated"
plt.scatter(
    X_emb[undiff_mask, 0], X_emb[undiff_mask, 1],
    c="lightgrey", s=10, alpha=0.3, linewidths=0
)

# Plot other cells
plt.scatter(
    X_emb[~undiff_mask, 0], X_emb[~undiff_mask, 1],
    c=cell_colors[~undiff_mask], s=10, alpha=0.5, linewidths=0
)

# Quiver: only keep locations with enough mass
plt.quiver(
    Xg[:, 0], Xg[:, 1],
    Vg[:, 0], Vg[:, 1],
    angles="xy",
    scale_units="xy",
    scale=1.5,
    width=0.005,
    headwidth=3,
    color="black",
    alpha=0.9
)

# Legend (excluding Undifferentiated)
handles = [
    plt.Line2D([], [], marker='o', linestyle='',
               color=custom_colors[label], label=label, markersize=6)
    for label in other_labels
]
plt.legend(handles=handles, fontsize=8, loc='center left',
           bbox_to_anchor=(1.02, 0.5), borderaxespad=0.)

plt.axis("equal")
plt.xticks([])
plt.yticks([])
plt.tight_layout()
plt.show()

In [ ]:
from scripts.VectorFieldGeometry import *

# 1) detect
fps = find_fixed_points_grid(emb.tps_vf, emb.X_emb,
                             tol_vec_percent=0.05,
                             tol_merge_percent=0.05,
                                       grid_size=100)

# 2) build the grid once (so Jacobian fits reuse it)
Xg, Vg = compute_velocity_on_grid(emb.X_emb, emb.tps_vf, grid_size=120)

# 3) loop through candidate fixed points
labels = []
for i, fp in enumerate(fps):
    J = jacobian_from_grid(emb.tps_vf, fp, Xg, Vg, radius_percent=0.2)
    label = classify_fixed_point(J)
    labels.append(label)
    print(f"FP{i}: {np.round(fp,3)}  →  {label}")

In [ ]:
from scripts.plotting import compute_velocity_on_grid
from matplotlib.lines import Line2D

# === 1. Prepare embedding and labels ===
X_emb = emb.X_emb
cell_labels = np.array(adata.obs["state_info"].values)

# === 2. Define custom colors ===
unique_labels = np.unique(cell_labels)
manual_color = {"Undifferentiated": "lightgrey"}
other_labels = [label for label in unique_labels if label != "Undifferentiated"]
tab10 = plt.get_cmap("tab10", len(other_labels))
tab_colors = {label: mcolors.to_hex(tab10(i)) for i, label in enumerate(other_labels)}
custom_colors = {**tab_colors, **manual_color}
cell_colors = np.array([custom_colors[label] for label in cell_labels])

# === 3. Compute velocity on grid ===
Xg_all, _, keep = compute_velocity_on_grid(X_emb, grid_size=30)
Xg = Xg_all[keep]
Vg = emb.tps_vf.predict(Xg)

# === 4. Plot scatter and quiver ===
plt.figure(figsize=(8, 8))

# Plot Undifferentiated first (background layer)
undiff_mask = cell_labels == "Undifferentiated"
plt.scatter(
    X_emb[undiff_mask, 0], X_emb[undiff_mask, 1],
    c="lightgrey", s=10, alpha=0.3, linewidths=0
)

# Plot other cells
plt.scatter(
    X_emb[~undiff_mask, 0], X_emb[~undiff_mask, 1],
    c=cell_colors[~undiff_mask], s=10, alpha=0.5, linewidths=0
)

# Quiver: only keep locations with enough mass
plt.quiver(
    Xg[:, 0], Xg[:, 1],
    Vg[:, 0], Vg[:, 1],
    angles="xy",
    scale_units="xy",
    scale=1.5,
    width=0.005,
    headwidth=3,
    color="black",
    alpha=0.9
)

# === 5. Select and sort fixed points (exclude index 1 and 6) ===
selected = [(i, fps[i], labels[i]) for i in range(len(fps)) if i not in {7}]
selected_sorted = sorted(selected, key=lambda x: (x[1][0], x[1][1]))  # sort by x then y

# === 6. Annotate sorted fixed points with red circles and white text ===
for new_idx, (old_idx, fp, _) in enumerate(selected_sorted, start=1):
    plt.scatter(fp[0], fp[1], color="red", s=120, zorder=4)
    plt.text(fp[0], fp[1], f"{new_idx}", color="white", fontsize=10,
             ha='center', va='center', weight='bold', zorder=5)

# === 7. Fixed point legend only ===
legend_labels = [f"{new_idx}: {labels[old_idx]}" for new_idx, (old_idx, _, _) in enumerate(selected_sorted, start=1)]
legend_handles_fp = [
    Line2D([0], [0], color='none', label=txt)
    for txt in legend_labels
]
plt.legend(handles=legend_handles_fp, fontsize=8, loc='center left',
           bbox_to_anchor=(1.02, 0.5), borderaxespad=0.)

# === 8. Remove plot box ===
ax = plt.gca()
for spine in ax.spines.values():
    spine.set_visible(False)

plt.axis("equal")
plt.xticks([])
plt.yticks([])
plt.tight_layout()
plt.show()

In [ ]:
emb.fit_gene_level_splines(dof_gene=100, dof_vf_gene=100)

In [ ]:
from scripts.GeneFlowAlignment import *
import seaborn as sns

gfa = GeneFlowAlignment(emb)
r2 = gfa.compute_gene_r2()

In [ ]:
# use correlation metrics
x = np.array(r2["expr_corr_gene"])  # Pearson r (expression)
y = np.array(r2["vel_corr_gene"])    # Cosine similarity (velocity)

# (optional) drop NaNs
mask = np.isfinite(x) & np.isfinite(y)
x, y = x[mask], y[mask]

# correlations
pearson = float(np.corrcoef(x, y)[0, 1])
try:
    from scipy.stats import spearmanr
    spearman = float(spearmanr(x, y, nan_policy="omit").correlation)
except Exception:
    spearman = np.nan

sns.set_style("whitegrid")
g = sns.jointplot(
    x=x, y=y,
    kind="scatter",
    color="steelblue",
    edgecolor="white",
    s=60, alpha=0.8,
    marginal_kws=dict(bins=40, fill=True)
)

# add a simple regression line
sns.regplot(x=x, y=y, scatter=False, ax=g.ax_joint, color="crimson", line_kws={"lw":2, "alpha":0.8})

g.set_axis_labels("Evaluation Metric (Expression: Pearson r)", "Evaluation Metric (Velocity: Pearson r)", fontsize=14)
g.ax_joint.axhline(0, color="grey", ls="--", lw=1, alpha=0.7)
g.ax_joint.axvline(0, color="grey", ls="--", lw=1, alpha=0.7)

plt.show()

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# inputs
cell_labels = np.array(adata.obs["state_info"].values)   # (N,)
cell_idx = None  # or subset if needed

# projections (from your compute_gene_r2)
X_proj = r2["X_proj"]
V_proj = r2["V_proj"]

# observed (match same cell_idx)
if cell_idx is None:
    cell_idx = np.arange(emb.X_raw.shape[0])

X_obs = emb.X_raw[cell_idx]
V_obs = emb.V_raw[cell_idx]

# Expression: L1 residuals
expr_resid = np.sum(X_obs - X_proj, axis=1)
expr_resid /= np.std(expr_resid)

# Velocity: cosine distance per cell (in [0, 2])
EPS = 1e-12
vel_cosdist = np.array([
    1.0 - (np.dot(vo, vp) / (np.linalg.norm(vo) * np.linalg.norm(vp) + EPS))
    for vo, vp in zip(V_obs, V_proj)
])
vel_cosdist = np.clip(vel_cosdist, 0.0, 2.0)  # numeric safety


# assemble df
df = pd.DataFrame({
    "cell_type": cell_labels[cell_idx],
    "expr_resid": expr_resid,      # keep as you wrote; swap to abs() if you want L1
    "vel_cosdist": vel_cosdist
})

# plots
plt.figure(figsize=(14, 5))
plt.subplot(1, 2, 1)
sns.violinplot(data=df, x="cell_type", y="expr_resid", inner="box", scale="width", cut=0, color="steelblue")
plt.title("Expression Residual (per cell)")  # add 'L1' if you switch to abs-sum
plt.xlabel("")
plt.xticks(rotation=45, ha="right")

plt.subplot(1, 2, 2)
sns.violinplot(data=df, x="cell_type", y="vel_cosdist", inner="box", scale="width", cut=0, color="mediumseagreen")
plt.title("Velocity Cosine Distance (per cell)")
plt.xlabel("")
plt.xticks(rotation=45, ha="right")
plt.axhline(1.0, color="grey", ls="--", lw=1, alpha=0.6)  # optional reference (orthogonal)

plt.tight_layout()
plt.show()

In [ ]:
r2.keys()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def plot_cell_and_radial_filtered(coord, cap_pct=99.0, quiver_scale=0.5, corr_thresh=0.2):
    coord = np.asarray(coord, float)

    # nearest cell
    dists = np.linalg.norm(X_emb - coord, axis=1)
    cell_idx = int(np.argmin(dists))
    print(f"[pick] cell {cell_idx}, coord={X_emb[cell_idx]}")

    # convert correlation lists to arrays and filter
    expr_corr = np.array(r2['expr_corr_gene'])
    vel_corr  = np.array(r2['vel_corr_gene'])
    good_mask = (expr_corr > corr_thresh) & (vel_corr > corr_thresh)
    print(f"Selected {good_mask.sum()} / {good_mask.size} genes")

    # gene gradients (filtered)
    G_grad = gfa.J[cell_idx][good_mask]         # (G_sel, 2)
    angles = np.arctan2(G_grad[:, 1], G_grad[:, 0])
    mags   = np.linalg.norm(G_grad, axis=1)

    # raw velocity values (filtered)
    vals = emb.V_raw[cell_idx, good_mask]
    cap = np.percentile(np.abs(vals), cap_pct)
    cap = max(cap, 1e-8)
    vals_clip = np.clip(vals, -cap, cap)

    # velocity vector for annotation
    vx, vy = emb.V_emb[cell_idx]
    vel_angle = np.arctan2(vy, vx)

    # plot
    fig = plt.figure(figsize=(14, 6))

    # left: embedding with annotated velocity
    ax1 = fig.add_subplot(1, 2, 1)
    ax1.scatter(X_emb[:, 0], X_emb[:, 1], c='lightgrey', s=10)
    ax1.quiver(Xg[:, 0], Xg[:, 1], Vg[:, 0], Vg[:, 1],
               angles="xy", scale_units="xy", scale=1.5,
               width=0.005, color="black", alpha=0.7)
    ax1.scatter(*X_emb[cell_idx], color="red", s=200, edgecolors='black')
    ax1.quiver(*X_emb[cell_idx], vx, vy,
               angles="xy", scale_units="xy",
               scale=quiver_scale, width=0.01, color="red")
    ax1.set_aspect("equal")
    ax1.set_title("Embedding + annotated velocity (emb.V_emb)")

    # right: radial plot (filtered genes, raw value color)
    ax2 = fig.add_subplot(1, 2, 2, projection='polar')
    sc = ax2.scatter(angles, mags, s=60, c=vals_clip, cmap='coolwarm',
                     vmin=-cap, vmax=cap, alpha=0.55)
    ax2.plot([vel_angle, vel_angle],
             [0, np.linalg.norm([vx, vy])],
             color='red', linewidth=2.2)
    ax2.set_theta_zero_location('E')
    ax2.set_theta_direction(1)
    cbar = plt.colorbar(sc, ax=ax2, pad=0.1)
    cbar.set_label(f'Raw V_raw (±{cap_pct}th pct), {good_mask.sum()} genes')

    plt.tight_layout()
    plt.show()

    return cell_idx

# example
plot_cell_and_radial_filtered([0, -2])

In [ ]:
plot_cell_and_radial_filtered([4.6, -0.1])
plot_cell_and_radial_filtered([5, 1])

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def plot_cell_and_radial_projected(coord, cap_pct=90.0, quiver_scale=0.5):
    coord = np.asarray(coord, float)

    # nearest cell
    dists = np.linalg.norm(X_emb - coord, axis=1)
    cell_idx = int(np.argmin(dists))
    print(f"[pick] cell {cell_idx}, coord={X_emb[cell_idx,]}")

    # projected velocity for q computation
    V_raw_cell = emb.V_raw[cell_idx, :]               
    J_embed = emb.tps_gene.compute_jacobians(np.array([X_emb[cell_idx,]]))
    J_embed = np.squeeze(J_embed, axis=0)
    if J_embed.shape == (V_raw_cell.shape[0], 2):
        u_t = J_embed.T @ V_raw_cell
    else:
        u_t = J_embed @ V_raw_cell

    # gene gradients and projected q_i
    G_grad = gfa.J[cell_idx]                          
    angles = np.arctan2(G_grad[:, 1], G_grad[:, 0])
    mags   = np.linalg.norm(G_grad, axis=1)
    q = G_grad @ u_t                                  

    # color cap
    cap = np.percentile(np.abs(q), cap_pct)
    cap = max(cap, 1e-8)
    q_clip = np.clip(q, -cap, cap)

    # velocity vector for annotation (from emb.V_emb)
    vx, vy = emb.V_emb[cell_idx]
    vel_angle = np.arctan2(vy, vx)
    # plot
    fig = plt.figure(figsize=(14, 6))

    # left: embedding with annotated velocity
    ax1 = fig.add_subplot(1, 2, 1)
    ax1.scatter(X_emb[:, 0], X_emb[:, 1], c='lightgrey', s=10)
    ax1.quiver(Xg[:, 0], Xg[:, 1], Vg[:, 0], Vg[:, 1],
               angles="xy", scale_units="xy", scale=1.5, width=0.005, color="black", alpha=0.7)
    ax1.scatter(*X_emb[cell_idx], color="red", s=200, edgecolors='black')
    ax1.quiver(*X_emb[cell_idx], vx, vy, angles="xy", scale_units="xy",
               scale=quiver_scale, width=0.01, color="red")
    ax1.set_aspect("equal")
    ax1.set_title("Embedding + annotated velocity (emb.V_emb)")

    # right: radial plot (colored by projected q_i)
    ax2 = fig.add_subplot(1, 2, 2, projection='polar')
    sc = ax2.scatter(angles, mags, s=22, c=q_clip, cmap='coolwarm',
                     vmin=-cap, vmax=cap, alpha=0.85)
    ax2.plot([vel_angle, vel_angle],
         [0, np.linalg.norm([vx, vy])],
         color='red', linewidth=2.2)
    ax2.set_theta_zero_location('E')
    ax2.set_theta_direction(1)
    cbar = plt.colorbar(sc, ax=ax2, pad=0.1)
    cbar.set_label(f'Projected change (±{cap_pct}th pct)')

    plt.tight_layout()
    plt.show()

    return cell_idx

# example
plot_cell_and_radial_projected([0, -2])

In [ ]:
plot_cell_and_radial_projected([4.6, -0.1])
plot_cell_and_radial_projected([5, 1])

In [ ]:
plot_cell_and_radial_projected([10, 7])

In [ ]:
from scripts.pseudotime import *

pseudotime_model = StochasticPseudotime(X_total=emb.X_emb, vector_field=emb.tps_vf, tps=emb.tps)
root = pseudotime_model.find_root(n_simulations_per_cell=5)
tau = pseudotime_model.compute_pseudotime(roots=root)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 1. Mask for monocytes
mask = (cell_labels == "Monocyte")

# 2. Velocity vectors
V = emb.V_emb[mask]  # shape: (n_monocytes, 2)

# 3. Gradient vectors for chosen gene
gene_name = "S100a9"
gene_idx = list(adata.var_names).index(gene_name)
G = gfa.J[mask, gene_idx, :]  # shape: (n_monocytes, 2)

# 4. Angles and magnitudes
angles_V = np.arctan2(V[:, 1], V[:, 0])
angles_G = np.arctan2(G[:, 1], G[:, 0])
angle_diff = angles_G - angles_V
angle_diff = (angle_diff + np.pi) % (2*np.pi) - np.pi  # wrap to [-π, π]

magnitudes = np.linalg.norm(G, axis=1)

# 5. Use y-coordinate from embedding for color
y_coord = emb.X_emb[mask, 1]
y_min, y_max = np.percentile(y_coord, [5, 95])  # clip outliers for better contrast
colors = np.clip(y_coord, y_min, y_max)

# 6. Radial plot
plt.figure(figsize=(6, 6))
ax = plt.subplot(111, polar=True)
sc = ax.scatter(angle_diff, magnitudes, alpha=0.8, s=15, cmap='Reds', c=colors)
ax.set_theta_zero_location("E")  # zero at velocity direction
ax.set_theta_direction(1)        # counterclockwise
ax.set_title(f"{gene_name} gradient vs velocity in monocytes")
plt.colorbar(sc, label='Embedding Y-coordinate')
plt.show()

In [ ]:
# 1. Mask for monocytes
mask = (cell_labels == "Monocyte")

# 2. Velocity vectors
V = emb.V_emb[mask]  # shape: (n_monocytes, 2)

# 3. Gradient vectors for chosen gene
gene_name = "Ctss"
gene_idx = list(adata.var_names).index(gene_name)
G = gfa.J[mask, gene_idx, :]  # shape: (n_monocytes, 2)

# 4. Angles and magnitudes
angles_V = np.arctan2(V[:, 1], V[:, 0])
angles_G = np.arctan2(G[:, 1], G[:, 0])
angle_diff = angles_G - angles_V
angle_diff = (angle_diff + np.pi) % (2*np.pi) - np.pi  # wrap to [-π, π]

magnitudes = np.linalg.norm(G, axis=1)

# 5. Use y-coordinate from embedding for color
y_coord = emb.X_emb[mask, 1]
y_min, y_max = np.percentile(y_coord, [5, 95])  # clip outliers for better contrast
colors = np.clip(y_coord, y_min, y_max)

# 6. Radial plot
plt.figure(figsize=(6, 6))
ax = plt.subplot(111, polar=True)
sc = ax.scatter(angle_diff, magnitudes, alpha=0.8, s=15, cmap='Reds', c=colors)
ax.set_theta_zero_location("E")  # zero at velocity direction
ax.set_theta_direction(1)        # counterclockwise
ax.set_title(f"{gene_name} gradient vs velocity in monocytes")
plt.colorbar(sc, label='Embedding Y-coordinate')
plt.show()

In [ ]:
# grad_norm: average L2 norm of each gene's Jacobian vector
grad_norm = np.linalg.norm(J_sub, axis=2).mean(axis=0)  # shape (n_genes,)

# Extract columns from stats_df
aligned_scores = stats_df["aligned_score"].values
orthogonal_scores = grad_norm  # or another orthogonal metric if you add it
gene_names = stats_df["gene_name"].values

# ── Thresholds ──
highlight_mask = (np.abs(aligned_scores) > 0.3) | (orthogonal_scores > 0.3)
highlight_x = aligned_scores[highlight_mask]
highlight_y = orthogonal_scores[highlight_mask]
highlight_genes = gene_names[highlight_mask]

# ── Plot ──
plt.figure(figsize=(6, 5))
plt.scatter(aligned_scores, orthogonal_scores, s=10, alpha=0.7, edgecolors='none')
plt.scatter(highlight_x, highlight_y, color='crimson', s=30, label='Highlighted genes')

# Draw labels for highlighted genes
for x, y, gene in zip(highlight_x, highlight_y, highlight_genes):
    plt.text(x, y, gene, fontsize=9, color='crimson', ha='left', va='bottom')

plt.axhline(0, color='grey', linestyle='--', linewidth=1)
plt.axvline(0, color='grey', linestyle='--', linewidth=1)
plt.xlabel("Flow-aligned score", fontsize=12)
plt.ylabel("Flow-orthogonal score", fontsize=12)
plt.title("Gene regime dynamics", fontsize=14)
plt.grid(True, linestyle='--', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
def plot_alignment_pval(stats_df, top_n=20, p_thresh=1.3):
    """
    Scatter: flow-aligned score vs -log10(p-value) for one stats_df.
    Highlights top_n genes by -log10(p).
    """
    x = stats_df["aligned_score"].values
    y = stats_df["-log10_pval"].values
    genes = stats_df["gene_name"].values

    # order by significance
    order = np.argsort(-y)[:top_n]
    hx, hy, hgenes = x[order], y[order], genes[order]

    plt.figure(figsize=(6,5))
    plt.scatter(x, y, s=10, alpha=0.6, edgecolors="none")
    plt.scatter(hx, hy, s=30, color="crimson")

    for xi, yi, g in zip(hx, hy, hgenes):
        plt.text(xi, yi, g, fontsize=8, color="crimson", ha="left", va="bottom")

    plt.axvline(0, color="grey", ls="--", lw=1)
    if p_thresh is not None:
        plt.axhline(p_thresh, color="grey", ls="--", lw=1, alpha=0.7)  # ~p=0.05 at 1.3
    plt.xlabel("Flow-aligned score")
    plt.ylabel("-log10(p-value)")
    plt.title("Flow alignment vs significance")
    plt.grid(ls="--", alpha=0.3)
    plt.tight_layout()
    plt.show()
    
# single
plot_alignment_pval(stats_df)

In [ ]:
# def plot_gene_regime_grid(stats_df, highlight_aligned=0.5, highlight_orth=0.5, ncols=3):
#     """
#     Plot flow-aligned vs orthogonal scores for each cell type in a grid.
#     """
#     cell_types = stats_df["cell_type"].unique()
#     ntypes = len(cell_types)
#     nrows = int(np.ceil(ntypes / ncols))
    
#     fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 4 * nrows), squeeze=False)

#     for ax, ctype in zip(axes.flat, cell_types):
#         df = stats_df[stats_df["cell_type"] == ctype]

#         aligned_scores = df["aligned_score"].values
#         orthogonal_scores = df["grad_norm"].values  # assuming grad_norm ≈ orthogonal magnitude

#         # Highlight mask
#         highlight_mask = (np.abs(aligned_scores) > highlight_aligned) | (orthogonal_scores > highlight_orth)
#         highlight_x = aligned_scores[highlight_mask]
#         highlight_y = orthogonal_scores[highlight_mask]
#         highlight_genes = df["gene_name"].values[highlight_mask]

#         # Scatter plot
#         ax.scatter(aligned_scores, orthogonal_scores, s=10, alpha=0.6, edgecolors='none')
#         ax.scatter(highlight_x, highlight_y, color='crimson', s=30)

#         # Labels for highlighted genes
#         for x, y, gene in zip(highlight_x, highlight_y, highlight_genes):
#             ax.text(x, y, gene, fontsize=7, color='crimson', ha='left', va='bottom')

#         ax.axhline(0, color='grey', linestyle='--', linewidth=1)
#         ax.axvline(0, color='grey', linestyle='--', linewidth=1)
#         ax.set_title(f"{ctype}", fontsize=12)
#         ax.set_xlabel("Flow-aligned score")
#         ax.set_ylabel("Flow-orthogonal score")

#     # Hide unused subplots if cell_types < nrows*ncols
#     for ax in axes.flat[ntypes:]:
#         ax.axis("off")

#     plt.tight_layout()
#     plt.show()


# # Example usage
# plot_gene_regime_grid(all_genes_stats_df)


In [ ]:
def plot_alignment_pval_grid(all_genes_stats_df, ncols=3, top_n=10, p_thresh=1.3):
    """
    Grid of scatter plots: one per cell_type in all_genes_stats_df.
    Highlights top_n genes in each panel by -log10(p).
    """
    cell_types = all_genes_stats_df["cell_type"].unique()
    n = len(cell_types)
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(5*ncols, 4*nrows), squeeze=False)

    for ax, ctype in zip(axes.flat, cell_types):
        df = all_genes_stats_df[all_genes_stats_df["cell_type"] == ctype]
        x = df["aligned_score"].values
        y = df["-log10_pval"].values
        genes = df["gene_name"].values

        # highlight top_n by significance
        order = np.argsort(-y)[:top_n]
        hx, hy, hgenes = x[order], y[order], genes[order]

        ax.scatter(x, y, s=8, alpha=0.6, edgecolors="none")
        ax.scatter(hx, hy, s=22, color="crimson")

        for xi, yi, g in zip(hx, hy, hgenes):
            ax.text(xi, yi, g, fontsize=7, color="crimson", ha="left", va="bottom")

        ax.axvline(0, color="grey", ls="--", lw=1)
        if p_thresh is not None:
            ax.axhline(p_thresh, color="grey", ls="--", lw=1, alpha=0.7)
        ax.set_title(f"{ctype}", fontsize=12)
        ax.set_xlabel("Flow-aligned score")
        ax.set_ylabel("-log10(p-value)")
        ax.grid(ls="--", alpha=0.3)

    # hide unused axes
    for ax in axes.flat[n:]:
        ax.axis("off")

    plt.tight_layout()
    plt.show()

# grid
plot_alignment_pval_grid(all_genes_stats_df, ncols=3)